# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore Record Sets and their fields using @id referencing
croissant_meta = dataset.metadata.to_json()

record_sets = croissant_meta.get('recordSet', [])
if not record_sets:
    print("No record sets listed in top-level metadata. Attempting to infer via mlcroissant...")
    record_sets_dict = dataset.record_sets
    record_sets = list(record_sets_dict.keys())
else:
    # recordSet is a list of dicts with '@id'
    record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]

print('Record Sets in dataset:')
for rs_id in record_sets:
    print(f"- Record Set @id: {rs_id}")

    # For each record set, show first few records and available columns/fields
    records = list(dataset.records(record_set=rs_id))
    if len(records):
        print(f"  Fields for {rs_id}: {list(records[0].keys())}")
        print(f"  Example record: {records[0]}")
    else:
        print(f"  No records found for {rs_id}.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities, including record sets and fields, are referenced by their `@id` fields.

In [ ]:
# Extract data from each record set using its @id
record_sets_ids = record_sets
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if not dataframes:
    print("No tabular record sets available.")
else:
    print("Columns for the extracted DataFrames:")
    for record_set_id, df in dataframes.items():
        print(f"Record set @id: {record_set_id} -> Columns: {df.columns.tolist()}")
    # Show head for the first record set
    first_rs = list(dataframes.keys())[0]
    print(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section applies to one tabular record set using its `@id`.

In [ ]:
# Select a numeric field for analysis, referenced by its @id (column name)
if dataframes:
    rs_analysis_id = list(dataframes.keys())[0]
    df = dataframes[rs_analysis_id]

    # Find a numeric column (heuristic: column name contains 'age', 'height', or matches known numeric fields)
    numeric_field_candidates = [c for c in df.columns if 'age' in c.lower() or 'height' in c.lower()] + df.select_dtypes(include=['number']).columns.tolist()
    numeric_field_candidates = list(set(numeric_field_candidates))
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        threshold = 20
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with field '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalize selected numeric field
        if len(filtered_df) > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized '{numeric_field_id}' for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try to group by a categorical field (column with 'infertility', 'hirsutism', 'phenotype', etc.)
            group_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['infertility', 'hirsutism', 'phenotype', 'variant', 'zygosity'])]
            if group_field_candidates:
                group_field = group_field_candidates[0]
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by '{group_field}':")
                print(grouped_df.head())
            else:
                print("No suitable group fields found for grouping.")
        else:
            print("No records above threshold for selected numeric field.")
    else:
        print("No suitable numeric columns found for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot distributions for the selected numeric field and relationships with group field, if available
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=10)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # Visualize relation with group_field if it exists
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(6,4))
        df.boxplot(column=numeric_field_id, by=group_field)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides a structured and transparent look at clinical and genetic features for three female patients with non-classical 21-hydroxylase deficiency.
- Data exploration demonstrates the utility of Croissant schemas in referencing data entities via their `@id` fields.
- The dataset highlights clinical heterogeneity and variant assessment, but is limited by sample size and scope—caution for generalization is warranted.
- This workflow can be extended to larger, multi-site datasets for deeper phenotype-genotype analyses.